# sysone quickstart: accuracy before and after training

Train a small decision model on synthetic support tickets and compare it with simple baselines on
held-out tickets, for each question type:

- **team** (choice): which team should handle the ticket (3 options)
- **severity** (score): how severe it is (4 levels)
- **refund** (noul): is the customer asking for a refund (yes / no)

It uses `tuning="head"`: the base model stays frozen and only the small decision head trains.
Expect a few minutes on a Mac; the first run also downloads the base model.

In [ ]:
# On Google Colab, or anywhere sysone isn't installed yet, install it from PyPI.
import importlib.util

if importlib.util.find_spec("sysone") is None:
    %pip install -q "sysone>=0.2"

In [ ]:
import random

from sysone import DecoderModel, EncoderModel, baselines, compare

# "qwen" -> Qwen/Qwen3-0.6B (about 1.2 GB download), "modernbert" -> answerdotai/ModernBERT-base (about 600 MB)
MODEL = "qwen"
Model = DecoderModel if MODEL == "qwen" else EncoderModel

## Data

Each ticket mentions a topic (which decides the team), a severity phrase, and in about half of
them a refund request. Rows are plain dicts: a `state` (the text), the `questions` to answer about
it, and the `expected` answers. `category` is optional; here it is the channel the ticket came in on.

In [ ]:
TEAMS = {
    "sales": ["a quote", "pricing", "a discount", "a contract renewal"],
    "operations": ["a delayed shipment", "the warehouse", "a tracking number", "a late delivery"],
    "servicing": ["an exchange", "a broken unit", "a repair", "returning the item"],
}
SEVERITY = ["none", "minor", "major", "critical"]
SEVERITY_WORDS = ["just a question", "a small issue", "a serious problem", "an emergency, everything is down"]
OPENERS = ["Hi, this is about", "Hello team, writing about", "Quick note regarding", "Urgent message about"]
REFUND_ASKS = ["I want my money back.", "Please refund the order.", "Can I get a refund?"]
CHANNELS = ["email", "chat", "phone"]

QUESTIONS = {
    "team": {"type": "choice", "instructions": "Which team should handle it?", "criteria": list(TEAMS)},
    "severity": {"type": "score", "instructions": "How severe is it?", "criteria": SEVERITY},
    "refund": {"type": "noul", "instructions": "Is the customer asking for a refund?"},
}


def make_ticket(rng, i):
    team = rng.choice(list(TEAMS))
    topic = rng.choice(TEAMS[team])
    level = rng.randrange(len(SEVERITY))
    refund = rng.random() < 0.5
    text = "Ticket %d: %s %s. It is %s." % (i, rng.choice(OPENERS), topic, SEVERITY_WORDS[level])
    if refund:
        text += " " + rng.choice(REFUND_ASKS)
    return {"state": text, "category": rng.choice(CHANNELS), "questions": QUESTIONS,
            "expected": {"team": team, "severity": level, "refund": refund}}


rng = random.Random(0)
train = [make_ticket(rng, i) for i in range(400)]
test = [make_ticket(rng, 400 + i) for i in range(150)]  # held out: never seen in training
print(len(train), "training tickets,", len(test), "test tickets")
train[0]

## Before training: baselines

`baselines` scores two simple guessers on the test tickets:
- **random guess**: every option equally likely;
- **average answer**: for each question, the average answer in the training data.

`compare` puts reports side by side; by default it shows accuracy per question type.

In [ ]:
results = baselines(test, train=train)
compare(results)

## Train

The backbone runs once over the training tickets and its outputs are cached; each epoch then
trains only the head. 10% of the tickets are held back for validation and early stopping.

In [ ]:
model = Model(tuning="head", batch_size=32)
model.fit(train)

## After training

In [ ]:
results["trained"] = model.evaluate(test)
compare(results)

**Metrics** (from `model.evaluate`, the same for hard and soft labels):
- **accuracy**: the model's most likely option is the right one (for soft labels: the raters'
  top answer). For a score question this means the exact level.
- **distance**: how far the predicted probabilities are from the label (0 = identical, 1 = completely
  different).
- **log_loss**: cross-entropy against the label; lower is better.
- **score_error**: for score questions, the gap between predicted and true expected level.
- **auc**: for yes/no questions, how well the model ranks yes above no (0.5 = chance).
- **brier**: squared distance between the predicted probabilities and the label; lower is better.
- **ece**: expected calibration error, how far "says X% sure" is from "right X% of the time";
  lower is better.

Probabilities are calibrated after training: a temperature per question type, fitted on the
validation slice, softens over-confident answers. This changes **log_loss**, **brier** and **ece**,
never **accuracy**, since the top answer stays the same.

In [ ]:
compare(results, by="overall")

## Try it yourself

Questions are passed at prediction time, so you can ask any of them about any text.
`probabilities` shows how sure the model is.

In [ ]:
ticket = "Ticket 9001: Urgent message about a broken unit. It is an emergency, everything is down. Can I get a refund?"
model.predict(ticket, QUESTIONS)

## Save and load

A saved model stores only the trained head and its configuration (a few hundred KB); the base
model is reloaded from the Hugging Face cache.

In [ ]:
model.save("quickstart-model")
reloaded = Model.load("quickstart-model", verbose=0)
assert reloaded.evaluate(test) == results["trained"]
print("reloaded model gives the same results")